# Exploratory Data Analysis (EDA)

**Financial Tweet Sentiment Classification — Nova IMS Text Mining 2025/2026**

This notebook performs a full exploratory analysis of the training and test datasets before any modelling.

Run top-to-bottom. All outputs are reproducible and idempotent.

In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import os, sys
from pathlib import Path

# Resolve project root whether running from repo/ or repo/notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Working directory: {os.getcwd()}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import numpy as np
from collections import Counter
import warnings
warnings.filterwarnings("ignore")
from wordcloud import WordCloud

from src.eda import DatasetAnalyzer, save_eda_plots, print_markdown_report
from src.config import (
    TRAIN_CSV_PATH, TEST_CSV_PATH, LABEL_NAMES,
    LABEL_PALETTE, URL_PATTERN, MENTION_PATTERN, CASHTAG_PATTERN,
    HASHTAG_PATTERN, DEFAULT_STOPWORDS,
)

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("Imports OK")


## 1. Load Datasets

In [ ]:
train_analyzer = DatasetAnalyzer(TRAIN_CSV_PATH, name="Train")
test_analyzer  = DatasetAnalyzer(TEST_CSV_PATH,  name="Test")

train_df = train_analyzer.df
test_df  = test_analyzer.df

print(f"Train shape : {train_df.shape}")
print(f"Test  shape : {test_df.shape}")
train_df.head(3)


## 2. Basic Statistics — Size & Integrity

In [ ]:
train_stats = train_analyzer.analyze_basic_stats()
test_stats  = test_analyzer.analyze_basic_stats()

stats_df = pd.DataFrame([train_stats, test_stats], index=["Train", "Test"])
print(stats_df.to_string())


## 3. Class Distribution (Train Only)

In [ ]:
dist = train_analyzer.analyze_class_distribution()
dist_df = pd.DataFrame(dist)

fig, ax = plt.subplots(figsize=(6, 4))
colors = [LABEL_PALETTE[r["label_name"]] for _, r in dist_df.iterrows()]
bars = ax.bar(dist_df["label_name"], dist_df["count"], color=colors, edgecolor="white", linewidth=1.2)

for bar, row in zip(bars, dist_df.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 60,
            f"{row.count:,}\n({row.percentage:.1f}%)", ha="center", va="bottom", fontsize=9)

ax.set_title("Sentiment Class Distribution — Training Set", fontsize=12, pad=12)
ax.set_xlabel("Sentiment Class")
ax.set_ylabel("Number of Tweets")
ax.set_ylim(0, dist_df["count"].max() * 1.18)
plt.tight_layout()
plt.show()

print("\nClass breakdown:")
for _, r in dist_df.iterrows():
    print(f"  {r['label_name']:8s}  {r['count']:5,}  ({r['percentage']:.2f}%)")


**Observation:** Severe class imbalance — Neutral dominates at ~64.7 %. All classifiers should use `class_weight='balanced'` or equivalent sample weighting.

## 4. Text Length Analysis

In [ ]:
len_stats = train_analyzer.analyze_text_lengths()

train_df["char_len"]  = train_df["text"].str.len()
train_df["token_len"] = train_df["text"].str.split().str.len()
train_df["class_name"] = train_df["label"].map(LABEL_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(data=train_df, x="char_len",  hue="class_name",
             multiple="layer", bins=35, palette=LABEL_PALETTE, ax=axes[0])
axes[0].set_title("Character Length Distribution")
axes[0].set_xlabel("Characters per Tweet")

sns.histplot(data=train_df, x="token_len", hue="class_name",
             multiple="layer", bins=25, palette=LABEL_PALETTE, ax=axes[1])
axes[1].set_title("Word (Token) Count Distribution")
axes[1].set_xlabel("Words per Tweet")

plt.suptitle("Tweet Length Distributions by Sentiment Class", y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

print("\nChar length summary:")
cl = len_stats["char_len"]
print(f"  mean={cl['mean']:.1f}  std={cl['std']:.1f}  min={cl['min']}  max={cl['max']}")
print("\nToken length summary:")
tl = len_stats["token_len"]
print(f"  mean={tl['mean']:.1f}  std={tl['std']:.1f}  min={tl['min']}  max={tl['max']}")


## 5. Twitter Metadata Artifact Rates

In [ ]:
art = train_analyzer.analyze_artifacts()

# Global rates table
global_rates = art["average_per_tweet"]
presence_pct  = art["presence_percentage"]

artifact_df = pd.DataFrame({
    "Artifact": ["URLs", "Mentions", "Cashtags", "Hashtags"],
    "Avg per Tweet": [global_rates["urls"], global_rates["mentions"],
                      global_rates["cashtags"], global_rates["hashtags"]],
    "% Tweets with Feature": [presence_pct["urls"], presence_pct["mentions"],
                               presence_pct["cashtags"], presence_pct["hashtags"]]
})
print(artifact_df.to_string(index=False))

# Bar chart by class
by_class = art["by_class"]
plot_rows = []
for cls, vals in by_class.items():
    for feat, key in [("URLs","urls_mean"),("Cashtags","cashtags_mean"),
                      ("Hashtags","hashtags_mean"),("Mentions","mentions_mean")]:
        plot_rows.append({"Class": cls, "Feature": feat, "Avg Count": vals[key]})

art_plot_df = pd.DataFrame(plot_rows)

plt.figure(figsize=(9, 4.5))
sns.barplot(data=art_plot_df, x="Feature", y="Avg Count", hue="Class", palette=LABEL_PALETTE)
plt.title("Average Metadata Artifact Counts by Sentiment Class", pad=12)
plt.ylabel("Average Count per Tweet")
plt.tight_layout()
plt.show()


**Observation:** URLs appear frequently in Neutral tweets (news sharing). Replacing URLs with `URL_PLACEHOLDER` preserves signal without expanding vocabulary.

## 6. Non-ASCII & Unicode Anomalies

In [ ]:
non_ascii = train_analyzer.analyze_non_ascii()
print(f"Non-ASCII tweets in train: {non_ascii['non_ascii_count']:,} ({non_ascii['non_ascii_percentage']:.2f}%)")
print("\nSample tweets with non-ASCII content:")
for ex in non_ascii["examples"]:
    print(f"  › {ex[:120]}")


## 7. Top Vocabulary by Sentiment Class

In [ ]:
top_words = train_analyzer.get_top_tokens(top_n=12)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
class_names = list(LABEL_NAMES.values())  # Bearish, Bullish, Neutral

for ax, cls_name in zip(axes, class_names):
    words = top_words["by_class"][cls_name]
    w_df  = pd.DataFrame(words)
    sns.barplot(data=w_df, x="count", y="word", color=LABEL_PALETTE[cls_name], ax=ax)
    ax.set_title(f"Top Vocab — {cls_name}", fontsize=11)
    ax.set_xlabel("Frequency")
    ax.set_ylabel("")

plt.suptitle("Top Informative Vocabulary per Sentiment Class", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()


## 8. Word Clouds (Train Set)

In [ ]:
import re

def make_corpus(df_subset):
    words = []
    for text in df_subset["text"]:
        t = URL_PATTERN.sub("", text)
        t = MENTION_PATTERN.sub("", t)
        t = CASHTAG_PATTERN.sub("", t)
        t = HASHTAG_PATTERN.sub("", t)
        t = re.sub(r"[^a-zA-Z\s]", "", t)
        words.extend([w.lower() for w in t.split()
                      if w.lower() not in DEFAULT_STOPWORDS and len(w) > 2])
    return " ".join(words)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (label_id, cls_name) in zip(axes, LABEL_NAMES.items()):
    corpus = make_corpus(train_df[train_df["label"] == label_id])
    wc = WordCloud(
        width=600, height=350, background_color="white",
        color_func=lambda *a, **kw: LABEL_PALETTE[cls_name],
        max_words=80, random_state=42
    ).generate(corpus)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(f"{cls_name} Tweets", fontsize=13, pad=8)

plt.suptitle("Word Clouds by Sentiment Class (Stopwords Removed)", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()


## 9. Save EDA Plots to Disk

In [ ]:
save_eda_plots(train_analyzer, output_dir="outputs/eda/")
print("Plots saved to outputs/eda/")


## 10. Full Markdown Report

In [ ]:
train_rpt = train_analyzer.generate_full_report(top_n_words=15)
test_rpt  = test_analyzer.generate_full_report(top_n_words=15)
print_markdown_report(train_rpt, test_rpt)


## 11. Key Findings & Design Decisions

| Finding | Decision |
| :--- | :--- |
| Severe imbalance: Neutral ~65%, Bullish ~20%, Bearish ~15% | Use `class_weight='balanced'` everywhere; primary metric = macro F1 |
| URLs present in ~70% of tweets — strong Neutral signal | Replace with `URL_PLACEHOLDER` (not remove) |
| Cashtags ($AAPL) are strong Bullish/Bearish signal | Keep cashtags in preprocessing pipeline (`cashtag_mode='keep'`) |
| Smart quotes, en-dashes, ellipses in ~N% of tweets | Apply `normalize_unicode_punctuation()` first in pipeline |
| Test set has no labels | Evaluation is val-only; final predictions → `outputs/pred_best.csv` |
| Mean tweet length ~20 tokens | DistilBERT max_length=64 is sufficient (avoids padding waste) |
